# Multi-Series (Panel Data) & Covariate Support
# 多序列（面板数据）与协变量支持

PipelineTS natively supports **panel data** (multiple time series) and **external covariates**.
PipelineTS 原生支持**面板数据**（多条时间序列）和**外部协变量**。

This tutorial covers:
本教程涵盖：

1. **Multi-series with `id_col` / 多序列 `id_col`**
2. **Per-series scaling / 每序列独立缩放**
3. **Multi-series in Pipeline & SmartRouter / 管道与智能路由器中的多序列**
4. **Known covariates (future) / 已知协变量（未来值）**
5. **Past covariates (historical only) / 历史协变量（仅历史值）**
6. **Combining multi-series + covariates / 组合多序列与协变量**
7. **Visualization of multi-series results / 多序列结果可视化**

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
LAGS = 12
PREDICT_N = 10

---
## Part I: Multi-Series (Panel Data)
## 第一部分：多序列（面板数据）

Use `id_col` to indicate which column identifies different series.

使用 `id_col` 指定标识不同序列的列。

### 1. Create Panel Data / 创建面板数据

In [ ]:
# Generate panel data with 3 series / 生成包含 3 条序列的面板数据
panel_dfs = []
for sid in ['Store_A', 'Store_B', 'Store_C']:
    n = 150
    dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
    base = np.random.uniform(30, 80)
    amp = np.random.uniform(5, 15)
    vals = base + amp * np.sin(np.linspace(0, 6 * np.pi, n)) + np.random.randn(n) * 2
    panel_dfs.append(pd.DataFrame({
        'date': dates,
        'value': vals,
        'store_id': sid
    }))

panel_data = pd.concat(panel_dfs, ignore_index=True)

print(f"Panel data shape / 面板数据形状: {panel_data.shape}")
print(f"Series count / 序列数量: {panel_data['store_id'].nunique()}")
print(f"\nSeries distribution / 序列分布:")
print(panel_data.groupby('store_id').size())
panel_data.head()

### 2. Visualize Multi-Series Data / 可视化多序列数据

In [ ]:
from PipelineTS.plot import plot_series

plot_series(
    panel_data,
    time_col='date', target_col='value',
    id_col='store_id',
    title='各门店销量趋势',
    lang='zh',
)

### 3. Single Model with `id_col` / 单模型使用 `id_col`

GBDT models handle `id_col` natively with per-series lag storage.

GBDT 模型原生处理 `id_col`，每序列独立存储滞后值。

In [ ]:
from PipelineTS.ml_model import TorchBoostingForestModel

model = TorchBoostingForestModel(
    time_col='date', target_col='value', lags=LAGS,
    quantile=0.9,
)
# Inject id_col into model config / 注入 id_col 到模型配置
model.all_configs['id_col'] = 'store_id'

model.fit(panel_data)
result = model.predict(PREDICT_N)

print(f"Prediction shape / 预测结果形状: {result.shape}")
print(f"Columns / 列名: {result.columns.tolist()}")
print(f"\nPredictions per series / 每序列预测数:")
print(result.groupby('store_id').size())
result.head()

### 4. ModelPipeline with Panel Data / ModelPipeline 面板数据

Pass `id_col` to `ModelPipeline`. Each series gets its own scaler.

将 `id_col` 传给 `ModelPipeline`。每条序列获得独立的缩放器。

In [ ]:
from PipelineTS.pipeline import ModelPipeline

pipeline = ModelPipeline(
    time_col='date',
    target_col='value',
    lags=LAGS,
    id_col='store_id',          # Enable panel mode / 启用面板模式
    include_models=['torch_boosting_forest', 'torch_bagging_forest'],
    quantile=0.9,
    cv=2,
)

leaderboard = pipeline.fit(panel_data)
print("Leaderboard / 排行榜:")
leaderboard

In [ ]:
# Predictions include store_id column / 预测结果包含 store_id 列
panel_pred = pipeline.predict(n=PREDICT_N)

print(f"Prediction shape / 预测结果形状: {panel_pred.shape}")
print(f"Series in prediction / 预测中的序列:")
print(panel_pred.groupby('store_id').size())
panel_pred.head(15)

### 5. SmartRouter with Panel Data / SmartRouter 面板数据

SmartRouter profiles the **longest series** as representative for routing decisions.

SmartRouter 使用**最长序列**作为路由决策的代表进行数据画像。

In [ ]:
from PipelineTS.pipeline import SmartRouter

router = SmartRouter(
    time_col='date',
    target_col='value',
    id_col='store_id',
    max_models=3,
)
router.fit(panel_data)

router_pred = router.predict(PREDICT_N)
print(f"SmartRouter prediction shape / SmartRouter 预测形状: {router_pred.shape}")
router_pred.head()

---
## Part II: Covariate Support
## 第二部分：协变量支持

PipelineTS supports two types of covariates:
PipelineTS 支持两类协变量：

| Type / 类型 | Description / 描述 | Models / 支持模型 |
|---|---|---|
| `known_covariates` | Future values known at prediction time (holidays, promotions) / 预测时已知的未来值 | GBDT, Prophet, AutoARIMA |
| `past_covariates` | Historical-only features (weather, sensors) / 仅历史特征 | GBDT |

### 6. Prepare Data with Covariates / 准备带协变量的数据

In [ ]:
# Generate data with covariates / 生成带协变量的数据
np.random.seed(42)
n = 200
dates = pd.date_range(start='2020-01-01', periods=n, freq='D')

# Covariates / 协变量
holiday = np.random.choice([0, 1], size=n, p=[0.9, 0.1])       # Known future / 已知未来值
promotion = np.random.choice([0, 1], size=n, p=[0.85, 0.15])   # Known future / 已知未来值
temperature = 15 + 10 * np.sin(np.linspace(0, 2 * np.pi, n)) + np.random.randn(n) * 2  # Past only / 仅历史

# Target depends on covariates / 目标值依赖协变量
values = (
    50 + 10 * np.sin(np.linspace(0, 6 * np.pi, n))
    + 8 * holiday          # Holiday boost / 节假日提升
    + 5 * promotion        # Promotion boost / 促销提升
    + 0.3 * temperature    # Temperature effect / 温度影响
    + np.random.randn(n) * 2
)

cov_data = pd.DataFrame({
    'date': dates,
    'value': values,
    'holiday': holiday,
    'promotion': promotion,
    'temperature': temperature,
})

print(f"Data shape / 数据形状: {cov_data.shape}")
print(f"Columns / 列名: {cov_data.columns.tolist()}")
cov_data.head()

### 7. Known Covariates in Pipeline / 管道中的已知协变量

In [ ]:
# Pipeline with known + past covariates / 带已知和历史协变量的管道
cov_pipeline = ModelPipeline(
    time_col='date',
    target_col='value',
    lags=LAGS,
    known_covariates=['holiday', 'promotion'],   # Future values known / 未来值已知
    past_covariates=['temperature'],              # Historical only / 仅历史
    include_models=['torch_boosting_forest', 'prophet'],
    quantile=0.9,
    cv=2,
)

# Training data must contain all covariate columns
# 训练数据必须包含所有协变量列
cov_leaderboard = cov_pipeline.fit(cov_data)
cov_leaderboard

In [ ]:
# At prediction time, provide future values of known covariates
# 预测时，提供已知协变量的未来值
future_cov = pd.DataFrame({
    'holiday':   [0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    'promotion': [1, 1, 0, 0, 0, 0, 0, 1, 1, 0],
})

cov_pred = cov_pipeline.predict(n=PREDICT_N, future_covariates=future_cov)
print(f"Prediction with covariates / 带协变量的预测:")
cov_pred

In [ ]:
from PipelineTS.plot import plot_forecast

plot_forecast(
    cov_data, cov_pred,
    time_col='date', target_col='value',
    history_tail=50,
    title='带协变量的预测（节假日 + 促销）',
    lang='zh',
)

### 8. Prediction Without Future Covariates / 不提供未来协变量的预测

If `future_covariates` is not provided but the model was trained with covariates, **zero placeholders** are used.

如果未提供 `future_covariates` 但模型训练时使用了协变量，将自动使用**零占位符**。

In [ ]:
# Predict without future_covariates (uses zeros) / 不提供未来协变量（使用零值）
cov_pred_no_future = cov_pipeline.predict(n=PREDICT_N)
print("Prediction without future covariates / 不带未来协变量的预测:")
cov_pred_no_future

### 9. SmartRouter with Covariates / SmartRouter 协变量

In [ ]:
cov_router = SmartRouter(
    time_col='date',
    target_col='value',
    known_covariates=['holiday', 'promotion'],
    past_covariates=['temperature'],
    max_models=3,
)
cov_router.fit(cov_data)

cov_router_pred = cov_router.predict(PREDICT_N, future_covariates=future_cov)
print("SmartRouter prediction with covariates / SmartRouter 协变量预测:")
cov_router_pred.head()

---
## Part III: Combined Multi-Series + Covariates
## 第三部分：多序列 + 协变量组合

In [ ]:
# Generate panel data with covariates / 生成带协变量的面板数据
combined_dfs = []
for sid in ['Region_North', 'Region_South']:
    n = 150
    dates = pd.date_range(start='2020-01-01', periods=n, freq='D')
    base = np.random.uniform(40, 70)
    holiday = np.random.choice([0, 1], size=n, p=[0.9, 0.1])
    vals = base + 8 * np.sin(np.linspace(0, 4 * np.pi, n)) + 6 * holiday + np.random.randn(n) * 2
    combined_dfs.append(pd.DataFrame({
        'date': dates,
        'value': vals,
        'region': sid,
        'holiday': holiday,
    }))

combined_data = pd.concat(combined_dfs, ignore_index=True)
print(f"Combined panel + covariate data / 组合面板+协变量数据: {combined_data.shape}")
combined_data.head()

In [ ]:
# Pipeline with both id_col and covariates
# 同时使用 id_col 和协变量的管道
combined_pipeline = ModelPipeline(
    time_col='date',
    target_col='value',
    lags=LAGS,
    id_col='region',
    known_covariates=['holiday'],
    include_models=['torch_boosting_forest'],
    quantile=0.9,
    cv=2,
)

combined_pipeline.fit(combined_data)

future_holidays = pd.DataFrame({'holiday': [0, 0, 1, 0, 0, 0, 0, 0, 1, 0]})
combined_pred = combined_pipeline.predict(n=PREDICT_N, future_covariates=future_holidays)

print(f"Combined prediction / 组合预测:")
print(f"Shape / 形状: {combined_pred.shape}")
print(f"Regions / 区域: {combined_pred['region'].unique().tolist()}")
combined_pred

## Summary / 总结

| Feature / 功能 | Parameter / 参数 | Description / 描述 |
|---|---|---|
| Multi-series / 多序列 | `id_col='series_id'` | Per-series scaling and prediction / 每序列独立缩放和预测 |
| Known covariates / 已知协变量 | `known_covariates=['col1']` | Future values known at prediction time / 预测时已知的未来值 |
| Past covariates / 历史协变量 | `past_covariates=['col2']` | Historical-only features / 仅历史特征 |
| Future covariates / 未来协变量 | `predict(future_covariates=df)` | Provide at prediction time / 预测时提供 |
| Zero fallback / 零值回退 | Automatic / 自动 | If future_covariates not provided / 未提供时使用零值 |
| Combined / 组合 | `id_col` + `known_covariates` | Multi-series + covariates together / 多序列与协变量组合 |